In [1]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import udtf

# Initialize Spark session
spark = SparkSession.builder \
    .appName("UDTFDemo") \
    .getOrCreate()

In [2]:
# Create a DataFrame with sample data
data = [
    (1, "apple,banana,orange"),
    (2, "car,bike,train"),
    (3, "cat,dog,elephant"),
    (4, None)  # Example with null input
]
columns = ["id", "items"]

df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show()

Original DataFrame:
+---+-------------------+
| id|              items|
+---+-------------------+
|  1|apple,banana,orange|
|  2|     car,bike,train|
|  3|   cat,dog,elephant|
|  4|               NULL|
+---+-------------------+



In [3]:
# Define a UDTF for splitting strings into rows
@udtf(returnType="id: int, item: string")
class SplitStrings:
    def eval(self, id: int, items: str):
        if items:  # Check if items is not None
            for item in items.split(","):  # Split items on commas
                yield id, item.strip()

In [4]:
# Register the UDTF for use in Spark SQL
spark.udtf.register("split_strings", SplitStrings)

# Register the DataFrame as a temporary SQL view
df.createOrReplaceTempView("input_table")

In [5]:
# Example: Using the UDTF directly in SQL
print("Split Items into Multiple Rows Using UDTF:")
spark.sql("""
    SELECT t.id, s.item
    FROM input_table t, LATERAL split_strings(t.id, t.items) s
""").show()

Split Items into Multiple Rows Using UDTF:
+---+--------+
| id|    item|
+---+--------+
|  1|   apple|
|  1|  banana|
|  1|  orange|
|  2|     car|
|  2|    bike|
|  2|   train|
|  3|     cat|
|  3|     dog|
|  3|elephant|
+---+--------+

